# Wizard OS: RAG tutor for Operating Systems education (Colab prototype)

Pipeline: **course files -> chunks -> embeddings (BGE) -> ChromaDB -> retrieval -> Qwen3 (4-bit) -> Gradio chat UI**

**Before you start**
1. `Runtime -> Change runtime type -> T4 GPU`
2. Put your course material (Silberschatz PDF, Linux notes, lab manuals, lecture PPTX, past papers) in a Google Drive folder, or upload it in Step 2.
3. Run the cells top to bottom.

## Step 1: Install and check GPU

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
!pip -q install chromadb sentence-transformers pypdf python-pptx langchain-text-splitters transformers accelerate bitsandbytes gradio
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - change runtime to T4 GPU")

CUDA available: True
GPU: Tesla T4


## Step 2: Add your knowledge base (Section 4 of the report)
Choose **one** option. Files are read from `KB_DIR`. Supported: `.pdf`, `.pptx`, `.txt`, `.md`, `.sh`, `.c`, `.py`.

In [3]:
import os
KB_DIR = "/content/knowledge_base"
os.makedirs(KB_DIR, exist_ok=True)

USE_DRIVE = True   # True: read from Google Drive folder. False: upload files by hand.

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    KB_DIR = "/content/drive/MyDrive/WizardOS_KB"   # <- create this folder in Drive and put your files in it
    os.makedirs(KB_DIR, exist_ok=True)
else:
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        with open(os.path.join(KB_DIR, name), "wb") as f:
            f.write(data)

print("Files found:")
for root, _, fs in os.walk(KB_DIR):
    for f in fs:
        print(" -", os.path.join(root, f))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files found:
 - /content/drive/MyDrive/WizardOS_KB/Shell Script.pdf
 - /content/drive/MyDrive/WizardOS_KB/Wizard_OS_QB_MCQ.pdf
 - /content/drive/MyDrive/WizardOS_KB/Wizard_OS_QB_Descriptive.pdf
 - /content/drive/MyDrive/WizardOS_KB/Wizard_OS_QB_LAB-EX.pdf
 - /content/drive/MyDrive/WizardOS_KB/Wizard_OS_QB_VIVA.pdf
 - /content/drive/MyDrive/WizardOS_KB/Deadlocks.pdf
 - /content/drive/MyDrive/WizardOS_KB/Disk scheduling.pdf
 - /content/drive/MyDrive/WizardOS_KB/Process-Scheduling.pdf
 - /content/drive/MyDrive/WizardOS_KB/Process Synchronization.pdf
 - /content/drive/MyDrive/WizardOS_KB/Memory-management.pdf


## Step 3: Parse documents and split into chunks (Section 5.1)

In [4]:
import glob, re
from collections import Counter
from pypdf import PdfReader
from pptx import Presentation
from langchain_text_splitters import RecursiveCharacterTextSplitter

CATS = [("mcq", ["mcq", "multiple"]), ("viva", ["viva"]),
        ("descriptive", ["descriptive", "long[ _-]?answer"]), ("lab", ["lab"])]

def categorize(path):
    s = os.path.relpath(path, KB_DIR).lower()
    for cat, keys in CATS:
        if any(re.search(rf"(^|[^a-z]){k}", s) for k in keys):
            return cat
    return "theory"

def detect_ext(path):
    name = os.path.basename(path)
    ext = name.lower().rsplit(".", 1)[-1] if "." in name else ""
    if ext in ("pdf", "pptx", "txt", "md", "sh", "c", "py"):
        return ext
    with open(path, "rb") as f:
        head = f.read(4)
    if head == b"%PDF":
        return "pdf"
    if head[:2] == b"PK":      # .pptx files are zip files
        return "pptx"
    return ext

def load_documents(folder):
    docs = []
    for path in sorted(glob.glob(f"{folder}/**/*", recursive=True)):
        if not os.path.isfile(path):
            continue
        name = os.path.basename(path)
        ext = detect_ext(path)
        cat = categorize(path)
        if ext == "pdf":
            for i, page in enumerate(PdfReader(path).pages, 1):
                t = page.extract_text() or ""
                if t.strip():
                    docs.append({"text": t, "source": name, "loc": f"p.{i}", "cat": cat})
        elif ext == "pptx":
            for i, slide in enumerate(Presentation(path).slides, 1):
                t = "\n".join(sh.text_frame.text for sh in slide.shapes if sh.has_text_frame)
                if t.strip():
                    docs.append({"text": t, "source": name, "loc": f"slide {i}", "cat": cat})
        elif ext in ("txt", "md", "sh", "c", "py"):
            with open(path, errors="ignore") as f:
                t = f.read()
            if t.strip():
                docs.append({"text": t, "source": name, "loc": "full file", "cat": cat})
        else:
            print("SKIPPED (unknown type):", name)
    return docs

docs = load_documents(KB_DIR)
print("Loaded", len(docs), "pages/slides/files")
print("\nFile -> category (check these are right):")
for s, c in sorted({(d["source"], d["cat"]) for d in docs}):
    print(f"  {c:12} {s}")

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = []
for d in docs:
    for j, piece in enumerate(splitter.split_text(d["text"])):
        chunks.append({"text": piece, "source": d["source"], "loc": d["loc"], "cat": d["cat"],
                       "id": f'{d["source"]}|{d["loc"]}|{j}'})
print("\nCreated", len(chunks), "chunks:", dict(Counter(c["cat"] for c in chunks)))

Loaded 398 pages/slides/files

File -> category (check these are right):
  theory       Deadlocks.pdf
  theory       Disk scheduling.pdf
  theory       Memory-management.pdf
  theory       Process Synchronization.pdf
  theory       Process-Scheduling.pdf
  theory       Shell Script.pdf
  descriptive  Wizard_OS_QB_Descriptive.pdf
  lab          Wizard_OS_QB_LAB-EX.pdf
  mcq          Wizard_OS_QB_MCQ.pdf
  viva         Wizard_OS_QB_VIVA.pdf

Created 532 chunks: {'theory': 415, 'descriptive': 33, 'lab': 29, 'mcq': 37, 'viva': 18}


## Step 4: Embed and store in ChromaDB (Sections 5.2 and 5.3)

In [5]:
import chromadb
from sentence_transformers import SentenceTransformer

EMB_MODEL = "BAAI/bge-large-en-v1.5"
emb = SentenceTransformer(EMB_MODEL, device="cuda")

vectors = emb.encode([c["text"] for c in chunks], batch_size=32,
                     normalize_embeddings=True, show_progress_bar=True)

client = chromadb.PersistentClient(path="/content/chroma_db")
try:
    client.delete_collection("wizard_os")
except Exception:
    pass
col = client.create_collection("wizard_os", metadata={"hnsw:space": "cosine"})

B = 500
for i in range(0, len(chunks), B):
    part = chunks[i:i+B]
    col.add(ids=[c["id"] for c in part],
            documents=[c["text"] for c in part],
            embeddings=vectors[i:i+B].tolist(),
            metadatas=[{"source": c["source"], "loc": c["loc"], "cat": c["cat"]} for c in part])
print("Stored", col.count(), "chunks")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Stored 532 chunks


## Step 5: Retrieval engine (Section 6.1)

In [6]:
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def retrieve(query, k=5, cats=None):
    qv = emb.encode([QUERY_PREFIX + query], normalize_embeddings=True)
    kwargs = {"where": {"cat": {"$in": cats}}} if cats else {}
    r = col.query(query_embeddings=qv.tolist(), n_results=k, **kwargs)
    out = []
    for doc, meta, dist in zip(r["documents"][0], r["metadatas"][0], r["distances"][0]):
        out.append({"text": doc, "source": meta["source"], "loc": meta["loc"],
                    "cat": meta.get("cat"), "score": round(1 - dist, 3)})
    return out

STYLE_CAT = {"Faculty: generate MCQs": "mcq",
             "Faculty: generate lab exercise": "lab",
             "Faculty: generate viva questions": "viva",
             "Faculty: generate descriptive questions": "descriptive"}

def retrieve_for_mode(question, mode, k=5):
    if mode == "Debug my code":
        hits = retrieve(question, k, cats=["theory", "lab"])
    elif mode in STYLE_CAT:
        hits = retrieve(question, 3, cats=[STYLE_CAT[mode]]) + retrieve(question, k, cats=["theory"])
    else:
        hits = retrieve(question, k, cats=["theory"])
    return hits or retrieve(question, k)   # fall back if a category is empty

for hit in retrieve_for_mode("Deadlocks", "Faculty: generate MCQs", k=3):
    print(hit["cat"], hit["score"], hit["source"], hit["loc"])

mcq 0.745 Wizard_OS_QB_MCQ.pdf p.9
mcq 0.718 Wizard_OS_QB_MCQ.pdf p.7
mcq 0.714 Wizard_OS_QB_MCQ.pdf p.9
theory 0.903 Deadlocks.pdf p.1
theory 0.903 Process-Scheduling.pdf p.43
theory 0.796 Deadlocks.pdf p.3


## Step 6: Load the base LLM in 4-bit (Section 6.2)
A free T4 has about 15 GB VRAM. `Qwen/Qwen3-8B` loads fast and is a safe default. To follow the report exactly, switch to `Qwen/Qwen3-14B` (4-bit is about 9 GB, but download is about 28 GB and generation is slower).

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen3-8B"      # or "Qwen/Qwen3-14B"

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb, device_map="auto")
print("Loaded", MODEL_NAME, "- VRAM used: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Loaded Qwen/Qwen3-8B - VRAM used: 7.4 GB


## Step 7: Tutor logic: modes, difficulty levels, grounded prompt (Section 3 objectives)

In [8]:
import re

SYSTEM = (
    "You are Wizard OS, a tutor for an undergraduate Operating Systems course. "
    "Answer using the COURSE CONTEXT provided. Prefer the terminology and algorithm versions used in the context. "
    "If the context does not cover the question, say so clearly, then give a short general answer marked as 'Outside course notes'. "
    "Never invent facts. Cite sources like [source, location]."
)

MODES = {
    "Explain a concept": "Explain the concept step by step, with a small worked example where useful.",
    "Debug my code": "The student has pasted code and/or an error. Identify the specific error, explain why it happens, show the corrected code, and give one tip to avoid it. Do not just rewrite everything silently.",
    "Faculty: generate MCQs": "Write exactly {count} multiple-choice questions numbered {first} to {last}. Each has 4 options (A-D), the correct answer marked, and a one-line explanation. Match the style of the past question papers in the context. Vary the topics and difficulty.",
    "Faculty: generate lab exercise": "Create exactly {count} different curriculum-aligned lab exercises numbered {first} to {last}. Each has: aim, problem statement, sample input/output, hints, and a reference solution.",
    "Faculty: generate viva questions": "Write exactly {count} viva questions numbered {first} to {last}, each with a short model answer. Order them from easy to hard.",
    "Faculty: generate descriptive questions": "Write exactly {count} descriptive (long-answer) exam questions numbered {first} to {last}. Each question must require a detailed explanation, for example 'Explain with a diagram...', 'Compare...', 'Solve and show each step...' or 'Discuss...'. Do not write one-line or factual-recall questions. After each question give the marks (use 10 unless the input says otherwise) and a model answer outline listing the key points and steps a full-mark answer must include. Match the style of the past question papers in the context. Vary the topics and difficulty.",
}

# mode -> (approx tokens per item, items per batch)
COUNT_MODES = {
    "Faculty: generate MCQs": (130, 10),
    "Faculty: generate lab exercise": (550, 2),
    "Faculty: generate viva questions": (110, 10),
    "Faculty: generate descriptive questions": (300, 4),
}

LEVELS = {
    "Beginner": "Use simple language, analogies, and avoid jargon. Define every term.",
    "Intermediate": "Assume the student knows basic OS terms. Be precise and concise.",
    "Advanced": "Be rigorous. Include complexity, edge cases, and trade-offs.",
}

def build_prompt(question, mode, level, hits, count=None, first=1, last=1, prior=""):
    ctx = "\n\n".join(f"[{h['source']}, {h['loc']}]\n{h['text']}" for h in hits)
    task = MODES[mode].format(count=count, first=first, last=last) if mode in COUNT_MODES else MODES[mode]
    extra = f"\n\nALREADY WRITTEN (do not repeat these):\n{prior[-3000:]}" if prior else ""
    return (f"COURSE CONTEXT:\n{ctx}\n\n"
            f"TASK: {task}\nDIFFICULTY: {LEVELS[level]}{extra}\n\n"
            f"TOPIC / INPUT:\n{question}")

def generate(user_msg, max_new_tokens):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_msg}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                             temperature=0.7, top_p=0.8)
    answer = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return re.sub(r"<think>.*?</think>", "", answer, flags=re.S).strip()

def ask(question, mode="Explain a concept", level="Intermediate", k=5, n=5, max_new_tokens=800):
    if mode not in COUNT_MODES:
        hits = retrieve_for_mode(question, mode, k)
        return generate(build_prompt(question, mode, level, hits), max_new_tokens), hits

    per_item, batch = COUNT_MODES[mode]
    n = max(1, min(int(n), 50))
    hits = retrieve_for_mode(question, mode, min(8, k + n // 5))   # more context for bigger requests
    parts, done = [], 0
    while done < n:
        m = min(batch, n - done)
        prompt = build_prompt(question, mode, level, hits, count=m, first=done + 1,
                              last=done + m, prior="\n\n".join(parts))
        parts.append(generate(prompt, 300 + m * per_item))
        done += m
    return "\n\n".join(parts), hits

ans, hits = ask("Round Robin scheduling", "Faculty: generate MCQs", "Intermediate", n=3)
print(ans)
print("\nSources:", sorted({(h['cat'], h['source'], h['loc']) for h in hits}))

1. What happens to the Round Robin scheduling algorithm when the time quantum is extremely large?  
A. It becomes equivalent to Shortest Job First  
B. It becomes equivalent to First-Come, First-Served  
C. It becomes equivalent to Priority Scheduling  
D. It becomes equivalent to Earliest Deadline First  
**Correct Answer: B**  
The algorithm degrades to FCFS as the time quantum approaches infinity [Process-Scheduling.pdf, p.22].  

2. In Round Robin scheduling, which of the following is a consequence of a very small time quantum?  
A. Long CPU bursts and infrequent I/O requests  
B. Short CPU bursts and frequent I/O requests  
C. Equal duration of CPU and I/O bursts  
D. Zero CPU burst requirements  
**Correct Answer: B**  
A very small time quantum leads to frequent context switches and more I/O requests [Process-Scheduling.pdf, p.22].  

3. Which of the following is a key characteristic of Round Robin scheduling?  
A. It uses priority-based scheduling  
B. It is preemptive  
C. It 

## Step 8: Chat interface (Gradio gives you a public link in Colab)

In [11]:
import gradio as gr

import traceback

def run(question, mode, level):
    if not question.strip():
        return "Please enter a question.", ""
    try:
        answer, hits = ask(question, mode, level)
        src = "\n".join(f"- {h['source']} ({h['loc']}), relevance {h['score']}" for h in hits)
        return answer, src
    except Exception:
        return "```\n" + traceback.format_exc() + "\n```", ""

with gr.Blocks(title="Wizard OS") as demo:
    gr.Markdown("# Wizard OS\nCurriculum-grounded Operating Systems tutor")
    with gr.Row():
        mode = gr.Dropdown(list(MODES), value="Explain a concept", label="Mode")
        level = gr.Dropdown(list(LEVELS), value="Intermediate", label="Difficulty")
    q = gr.Textbox(lines=6, label="Question, topic, or pasted code/error")
    btn = gr.Button("Ask Wizard OS", variant="primary")
    a = gr.Markdown(label="Answer")
    s = gr.Markdown(label="Sources")
    btn.click(run, [q, mode, level], [a, s])

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://99ae588501aa54fc9f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Step 9: Save your vector database so you do not rebuild it every session

In [14]:
import shutil
shutil.copytree("/content/chroma_db", "/content/drive/MyDrive/WizardOS_chroma_db", dirs_exist_ok=True)
print("Saved to Drive. Next session: mount Drive, load with chromadb.PersistentClient(path=...), skip Steps 3-4.")

Saved to Drive. Next session: mount Drive, load with chromadb.PersistentClient(path=...), skip Steps 3-4.


In [13]:
import ipywidgets as w
import traceback
from IPython.display import display, Markdown, clear_output

mode = w.Dropdown(options=list(MODES), value="Explain a concept", description="Mode:")
level = w.Dropdown(options=list(LEVELS), value="Intermediate", description="Level:")
count = w.BoundedIntText(value=5, min=1, max=50, description="How many:",
                         tooltip="Used by the Faculty modes only")
note = w.HTML("<i>'How many' applies to the Faculty modes (MCQs, viva, descriptive, lab).</i>")
q = w.Textarea(placeholder="Question, topic, or pasted code/error. For Faculty modes, type the topic, e.g. Round Robin scheduling",
               layout=w.Layout(width="100%", height="120px"))
btn = w.Button(description="Ask Wizard OS", button_style="primary")
out = w.Output()

def on_click(_):
    with out:
        clear_output()
        print("Thinking... large requests can take several minutes.")
        try:
            ans, hits = ask(q.value, mode.value, level.value, n=count.value)
        except Exception:
            clear_output()
            traceback.print_exc()
            return
        clear_output()
        display(Markdown(ans))
        src = "\n".join(f"- {h['source']} ({h['loc']}), {h['cat']}, relevance {h['score']}" for h in hits)
        display(Markdown("**Sources:**\n" + src))

btn.on_click(on_click)
display(mode, level, count, note, q, btn, out)

Dropdown(description='Mode:', options=('Explain a concept', 'Debug my code', 'Faculty: generate MCQs', 'Facult…

Dropdown(description='Level:', index=1, options=('Beginner', 'Intermediate', 'Advanced'), value='Intermediate'…

BoundedIntText(value=5, description='How many:', max=50, min=1)

HTML(value="<i>'How many' applies to the Faculty modes (MCQs, viva, descriptive, lab).</i>")

Textarea(value='', layout=Layout(height='120px', width='100%'), placeholder='Question, topic, or pasted code/e…

Button(button_style='primary', description='Ask Wizard OS', style=ButtonStyle())

Output()

## Next steps (the "fine-tuned" part of the title)
1. Collect 300-1000 Q&A pairs from your lecture notes and past papers (JSONL: instruction, context, response).
2. Fine-tune with LoRA using `peft` + `trl` (SFTTrainer) on the 4-bit model (QLoRA), then reload the adapter in Step 6.
3. Evaluate: 30 test questions, compare base vs RAG vs RAG + fine-tuned for correctness and hallucination rate.
4. For the Streamlit UI from the report, run it outside Colab (local machine) or expose it through a tunnel; Gradio is easier inside Colab.

In [8]:
import pandas as pd, json, random, re, os
random.seed(42)

TRAIN_DIR = "/content/drive/MyDrive/WizardOS_Training"
XLSX = f"{TRAIN_DIR}/Wizard_OS_Training_Review.xlsx"

df = pd.read_excel(XLSX).fillna("")
df.columns = [c.strip() for c in df.columns]
print(df["Type"].value_counts().to_dict())
notes = df["Your corrections / notes"].astype(str).str.strip().ne("").sum()
print("Rows with text in 'Your corrections / notes':", notes, "(these notes are NOT used, edit the Draft answer cell instead)")

def clean(s):
    return re.sub(r"[ \t]+", " ", str(s)).strip()

bank = {}
for t in ["MCQ", "Viva", "Descriptive", "Lab"]:
    sub = df[df["Type"] == t]
    bank[t] = [{"sec": clean(r["Section"]), "q": str(r["Question"]).strip(), "a": str(r["Draft answer"]).strip()}
               for _, r in sub.iterrows() if str(r["Draft answer"]).strip()]

# hold out every 10th item of each type as a test set (never used for training)
train_items, test_items = {}, {}
for t, items in bank.items():
    train_items[t] = [x for i, x in enumerate(items) if i % 10 != 9]
    test_items[t]  = [x for i, x in enumerate(items) if i % 10 == 9]

def make(topic, mode, level, response, count=None, first=1, last=1):
    hits = retrieve_for_mode(topic, mode, 3)
    prompt = build_prompt(topic, mode, level, hits, count=count, first=first, last=last)
    return {"mode": mode, "level": level, "topic": topic, "count": count, "prompt": prompt, "response": response}

def groups(items, size):
    out, cur = [], []
    for x in items:
        if cur and (x["sec"] != cur[0]["sec"] or len(cur) == size):
            out.append(cur); cur = []
        cur.append(x)
    if cur: out.append(cur)
    return out

def numbered(g, fmt):
    return "\n\n".join(f"{i}. " + fmt(x) for i, x in enumerate(g, 1))

M_MCQ, M_VIVA, M_DESC, M_LAB, M_EXP = ("Faculty: generate MCQs", "Faculty: generate viva questions",
    "Faculty: generate descriptive questions", "Faculty: generate lab exercise", "Explain a concept")

def build(items_by_type, with_groups=True):
    ex = []
    # 1. Student asks a question -> model answer (viva + descriptive banks)
    for x in items_by_type["Viva"]:
        ex.append(make(clean(x["q"]), M_EXP, "Intermediate", x["a"]))
    for x in items_by_type["Descriptive"]:
        ex.append(make(clean(x["q"]), M_EXP, "Advanced", x["a"]))
    # 2. Faculty generation
    mcq_fmt = lambda x: x["q"] + "\n" + x["a"]
    for x in items_by_type["MCQ"]:
        ex.append(make(x["sec"], M_MCQ, "Intermediate", numbered([x], mcq_fmt), count=1, first=1, last=1))
    viva_fmt = lambda x: clean(x["q"]) + "\nModel answer: " + x["a"]
    if with_groups:
        for g in groups(items_by_type["MCQ"], 5):
            if len(g) > 1: ex.append(make(g[0]["sec"], M_MCQ, "Intermediate", numbered(g, mcq_fmt), count=len(g), first=1, last=len(g)))
        for g in groups(items_by_type["Viva"], 5):
            if len(g) > 1: ex.append(make(g[0]["sec"], M_VIVA, "Intermediate", numbered(g, viva_fmt), count=len(g), first=1, last=len(g)))
    for x in items_by_type["Viva"]:
        ex.append(make(x["sec"], M_VIVA, "Intermediate", numbered([x], viva_fmt), count=1, first=1, last=1))
    for x in items_by_type["Descriptive"]:
        body = clean(x["q"]) + "\nModel answer outline:\n" + x["a"]
        ex.append(make(x["sec"], M_DESC, "Advanced", "1. " + body, count=1, first=1, last=1))
    for x in items_by_type["Lab"]:
        title = x["q"].split(":")[0].strip()
        body = clean(x["q"]) + "\nReference solution:\n```\n" + x["a"] + "\n```"
        ex.append(make(title, M_LAB, "Intermediate", "1. " + body, count=1, first=1, last=1))
    return ex

train_ex = build(train_items, with_groups=True)
test_ex  = build(test_items, with_groups=False)
random.shuffle(train_ex)

for name, data in (("train", train_ex), ("test", test_ex)):
    with open(f"{TRAIN_DIR}/{name}.jsonl", "w", encoding="utf-8") as f:
        for e in data: f.write(json.dumps(e, ensure_ascii=False) + "\n")

from collections import Counter
print("train examples:", len(train_ex), dict(Counter(e["mode"] for e in train_ex)))
print("test examples :", len(test_ex),  dict(Counter(e["mode"] for e in test_ex)))
print("Saved train.jsonl and test.jsonl to", TRAIN_DIR)
print("\n--- sample training example (mode: %s) ---" % train_ex[0]["mode"])
print(train_ex[0]["prompt"][-500:], "\n=> RESPONSE:\n", train_ex[0]["response"][:500])

{'MCQ': 120, 'Viva': 120, 'Descriptive': 120, 'Lab': 120}
Rows with text in 'Your corrections / notes': 0 (these notes are NOT used, edit the Draft answer cell instead)
train examples: 696 {'Explain a concept': 216, 'Faculty: generate MCQs': 132, 'Faculty: generate descriptive questions': 108, 'Faculty: generate viva questions': 132, 'Faculty: generate lab exercise': 108}
test examples : 72 {'Explain a concept': 24, 'Faculty: generate MCQs': 12, 'Faculty: generate viva questions': 12, 'Faculty: generate descriptive questions': 12, 'Faculty: generate lab exercise': 12}
Saved train.jsonl and test.jsonl to /content/drive/MyDrive/WizardOS_Training

--- sample training example (mode: Explain a concept) ---
     Pi enters critical section only if:
                      turn = I 
   and turn cannot be both 0 and 1 at the same time
▪ What about the Progress requirement?
             FOObar
▪ What about the Bounded-waiting requirement?

TASK: Explain the concept step by step, with a small worke

In [ ]:
import json, glob, math, torch
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

TRAIN_DIR = "/content/drive/MyDrive/WizardOS_Training"
MAXLEN = 2048
EPOCHS = 2

rows = [json.loads(l) for l in open(f"{TRAIN_DIR}/train.jsonl", encoding="utf-8")]

def encode(ex):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": ex["prompt"]}]
    p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    p_ids = tok(p, add_special_tokens=False)["input_ids"]
    r_ids = tok(ex["response"] + "<|im_end|>", add_special_tokens=False)["input_ids"]
    ids = p_ids + r_ids
    if len(ids) > MAXLEN:
        return None
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": [-100] * len(p_ids) + r_ids}

data = [e for e in (encode(r) for r in rows) if e]
print(f"Kept {len(data)} of {len(rows)} examples (<= {MAXLEN} tokens); longest = {max(len(d['input_ids']) for d in data)}")

class DS(torch.utils.data.Dataset):
    def __len__(self): return len(data)
    def __getitem__(self, i): return data[i]

if not hasattr(model, "peft_config"):          # only attach the adapter once
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    model = get_peft_model(model, lora)
model.print_trainable_parameters()

total_steps = math.ceil(len(data) / 8) * EPOCHS
args = TrainingArguments(
    output_dir=f"{TRAIN_DIR}/checkpoints", per_device_train_batch_size=1, gradient_accumulation_steps=8,
    num_train_epochs=EPOCHS, learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=max(1, int(0.05 * total_steps)),
    fp16=True, logging_steps=5, save_steps=40, save_total_limit=2, optim="paged_adamw_8bit",
    report_to="none", remove_unused_columns=False)
trainer = Trainer(model=model, args=args, train_dataset=DS(),
                  data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100))

resume = bool(glob.glob(f"{TRAIN_DIR}/checkpoints/checkpoint-*"))
trainer.train(resume_from_checkpoint=resume or None)

model.save_pretrained(f"{TRAIN_DIR}/wizard_os_lora")
tok.save_pretrained(f"{TRAIN_DIR}/wizard_os_lora")
model.config.use_cache = True
model.eval()
print("Adapter saved to", f"{TRAIN_DIR}/wizard_os_lora")

Kept 679 of 696 examples (<= 2048 tokens); longest = 2048
trainable params: 43,646,976 || all params: 8,234,382,336 || trainable%: 0.5301


Step,Training Loss
5,1.866079
10,1.256987
15,1.126495
20,1.162342
25,1.027242
30,0.961477
35,0.881152
